In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi
!nvcc --version
!python --version
!free -g
!df -h /kaggle/working /kaggle/temp

Sun Sep 20 07:44:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q uv
%cd /kaggle/working
!git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
%cd /kaggle/working/openpi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 66.0 MB/s eta 0:00:00:00:0100:01
/kaggle/working
Cloning into 'openpi'...
remote: Enumerating objects: 1667, done.
remote: Total 1667 (delta 0), reused 0 (delta 0), pack-reused 1667 (from 1)
Receiving objects: 100% (1667/1667), 24.71 MiB | 42.53 MiB/s, done.
Resolving deltas: 100% (883/883), done.
Submodule 'third_party/aloha' (https://github.com/Physical-Intelligence/aloha.git) registered for path 'third_party/aloha'
Submodule 'third_party/libero' (https://github.com/Lifelong-Robot-Learning/LIBERO.git) registered for path 'third_party/libero'
Cloning into '/kaggle/working/openpi/third_party/aloha'...
remote: Enumerating objects: 259, done.        
remote: Counting objects: 100% (91/91), done.        
remote: Compressing objects: 100% (67/67), done.        
remote: Total 259 (delta 43), reused 55 (delta 20), pack-reused 168 (from 1)        
Receiving objects: 100% (259/259), 42.74 MiB | 29.57 MiB/s, done.
Resolving deltas: 100% (

In [3]:
!GIT_LFS_SKIP_SMUDGE=1 uv sync
!GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

Using CPython 3.11.16
Creating virtual environment at: .venv
Resolved 279 packages in 1ms
Prepared 240 packages in 42.04s                                          
░░░░░░░░░░░░░░░░░░░░ [0/240] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 240 packages in 23.27s                            
 + absl-py==2.3.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.12.4
 + aiosignal==1.3.2
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + asttokens==3.0.0
 + attrs==25.3.0
 + augmax==0.4.1
 + av==17.0.0
 + beartype==0.19.0
 + beautifulsoup4==4.13.4
 + blinker==1.9.0
 + cachetools==5.5.2
 + certifi==2025.4.26
 + cffi==1.17.1
 + cfgv==3.4.0
 + charset-normalize

In [4]:
!uv run python -c "import jax; print(jax.devices())"

[CudaDevice(id=0), CudaDevice(id=1)]


In [10]:
%%writefile /kaggle/working/test_infer.py
from openpi.training import config as _config
from openpi.policies import policy_config
from openpi.policies import libero_policy
from openpi.shared import download

cfg = _config.get_config("pi05_libero")
ckpt = download.maybe_download("gs://openpi-assets/checkpoints/pi05_libero")
policy = policy_config.create_trained_policy(cfg, ckpt)

example = libero_policy.make_libero_example()
for k, v in example.items():
    print(k, "->", getattr(v, "shape", v), getattr(v, "dtype", ""))

result = policy.infer(example)
print("action shape:", result["actions"].shape)

Overwriting /kaggle/working/test_infer.py


In [16]:
!cd /kaggle/working/openpi && uv run python /kaggle/working/test_infer.py

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://openpi-assets/checkpoints/pi05_libero/params/manifest.ocdbt...
Copying gs://openpi-assets/checkpoints/pi05_libero/assets/physical-intelligence/libero/norm_stats.json...
Copying gs://openpi-assets/checkpoints/pi05_libero/params/_METADATA...          
Copying gs://openpi-assets/checkpoints/pi05_libero/params/ocdbt.process_0/d/bda87d9791f23df771cd2d15293780cc...
Copying gs://openpi-assets/checkpoints/pi05_libero/params/ocdbt.process_0/d/0eaaecefaa9720d30a32cc56e65fd345...
Copying gs://openpi-assets/checkpoints/pi05_libero/params/array_metadatas/process_0...
Copying gs://openpi-assets/checkpoints/pi05_libero/params/ocdbt.process_0/d/2b6985f48e9da86f68627a7608c5bc25...
Copying gs://openpi-assets/checkpoints/pi05_libero/para

In [17]:
!sed -n '1,80p' /kaggle/working/openpi/src/openpi/policies/libero_policy.py

import dataclasses

import einops
import numpy as np

from openpi import transforms
from openpi.models import model as _model


def make_libero_example() -> dict:
    """Creates a random input example for the Libero policy."""
    return {
        "observation/state": np.random.rand(8),
        "observation/image": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
        "observation/wrist_image": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
        "prompt": "do something",
    }


def _parse_image(image) -> np.ndarray:
    image = np.asarray(image)
    if np.issubdtype(image.dtype, np.floating):
        image = (255 * image).astype(np.uint8)
    if image.shape[0] == 3:
        image = einops.rearrange(image, "c h w -> h w c")
    return image


@dataclasses.dataclass(frozen=True)
class LiberoInputs(transforms.DataTransformFn):
    """
    This class is used to convert inputs to the model to the expected format. It is used for both training and inference.

  

In [21]:
%%writefile /kaggle/working/test_infer.py
import time
from openpi.training import config as _config
from openpi.policies import policy_config, libero_policy
from openpi.shared import download

cfg = _config.get_config("pi05_libero")
ckpt = download.maybe_download("gs://openpi-assets/checkpoints/pi05_libero")
policy = policy_config.create_trained_policy(cfg, ckpt)
example = libero_policy.make_libero_example()

policy.infer(example)          # warm-up: JIT compilation happens here
t0 = time.time()
for _ in range(10):
    policy.infer(example)
print((time.time() - t0) / 10, "s per call")

Overwriting /kaggle/working/test_infer.py


In [22]:
!cd /kaggle/working/openpi && uv run python /kaggle/working/test_infer.py

1.883927583694458 s per call
